In [7]:
!pip install navec

In [3]:
!pip uninstall pymorphy2 -y
!pip install pymorphy2==0.8

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 21.5 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=d09f12cb164af4a70899a3f3a037967a7bf8c97b6732f0cef4edf990dcff2f6c
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built docopt


In [4]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 16.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [5]:
!pip install natasha

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.6 MB/s eta 0:00:00
  Created wheel for intervaltree: filename=intervaltree-3.1.0-py2.py3-none-any.whl size=26097 sha256=b255419fa47e7ae1ab0c03f82eb4c7b16daef9135dab440614b18b51598fb361
  Stored in directory: /root/.cache/pip/wheels/31/d7/d9/eec6891f78cac19a693bd40ecb8365d2f4613318c145ec9816
Successfully built intervaltree


In [9]:
!pip uninstall pandas -y
!pip install --no-cache-dir pandas

Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 138.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 174.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [10]:
!pip install pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 25.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3


In [6]:
import pandas as pd
import numpy as np
import re
import pymorphy2
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim
import navec
from gensim.models import Word2Vec

In [8]:

RANDOM_STATE = 42

url = 'https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz'
df = pd.read_csv(url, compression='gzip', usecols=['title', 'text', 'topic'])

In [9]:

df = df[['text', 'topic']].dropna()

df = df.sample(50000, random_state=RANDOM_STATE)

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яё]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess_text)

In [10]:
morph = pymorphy2.MorphAnalyzer()

def lemmatize_text(text):
    return " ".join([morph.parse(word)[0].normal_form for word in text.split()])

df['lemmatized_text'] = df['clean_text'].apply(lemmatize_text)

In [11]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['lemmatized_text'], df['topic'], test_size=0.2, stratify=df['topic'], random_state=RANDOM_STATE)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.25, stratify=train_labels, random_state=RANDOM_STATE)  # 0.25 * 0.8 = 0.2

len(train_texts), len(val_texts), len(test_texts)

(30000, 10000, 10000)

In [12]:
from gensim.models import Word2Vec

train_sentences = [text.split() for text in train_texts]

w2v_model = Word2Vec(sentences=train_sentences,
                      vector_size=100,  # Размерность
                      window=5,  # Окно контекста
                      min_count=5,  # Игнорируем слова, встречающиеся реже 5 раз
                      workers=4,  # Количество потоков
                      sg=1,  # Используем Skip-gram
                      seed=RANDOM_STATE)

w2v_model.save("word2vec_model.model")

similar_words = w2v_model.wv.most_similar("животное", topn=5)
similar_words

[('насекомое', 0.7548351287841797),
 ('растение', 0.7383272051811218),
 ('птица', 0.721799910068512),
 ('дикий', 0.7159053683280945),
 ('примат', 0.7124671936035156)]

In [13]:
similar_words = w2v_model.wv.most_similar("животное", topn=5)

odd_one_out = w2v_model.wv.doesnt_match(["корова", "овца", "картина", "лощадь"])

similar_words, odd_one_out

([('насекомое', 0.7548351287841797),
  ('растение', 0.7383272051811218),
  ('птица', 0.721799910068512),
  ('дикий', 0.7159053683280945),
  ('примат', 0.7124671936035156)],
 'картина')

In [14]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar -O navec.tar

navec_model = navec.Navec.load("navec.tar")

word = "животное"
if word in navec_model:
    navec_vector = navec_model[word]
    print(f"Вектор для '{word}': {navec_vector[:5]}")
else:
    print(f"Слово '{word}' отсутствует в модели")

--2025-03-14 15:59:24--  https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53012480 (51M) [application/x-tar]
Saving to: ‘navec.tar’

navec.tar           100%[===================>]  50.56M  16.1MB/s    in 3.5s    

2025-03-14 15:59:28 (14.6 MB/s) - ‘navec.tar’ saved [53012480/53012480]

Вектор для 'животное': [-0.3389153   0.01224285  0.17503086  0.47832823 -0.24603093]


In [15]:
rusvectores_url = "http://vectors.nlpl.eu/repository/20/220.zip"

!wget -O rusvectores.zip $rusvectores_url
!unzip rusvectores.zip -d rusvectores

model_path = "rusvectores/model.bin"
rusvectores_model = gensim.models.KeyedVectors.load_word2vec_format(model_path, binary=True)

word = "животное"
if word in rusvectores_model:
    rusvectores_vector = rusvectores_model[word]
else:
    rusvectores_vector = None

rusvectores_vector

--2025-03-14 15:59:29--  http://vectors.nlpl.eu/repository/20/220.zip
Resolving vectors.nlpl.eu (vectors.nlpl.eu)... 129.240.189.200, 2001:700:112::200
Connecting to vectors.nlpl.eu (vectors.nlpl.eu)|129.240.189.200|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 638171816 (609M) [application/zip]
Saving to: ‘rusvectores.zip’

rusvectores.zip     100%[===================>] 608.61M  19.6MB/s    in 32s     

2025-03-14 16:00:02 (19.0 MB/s) - ‘rusvectores.zip’ saved [638171816/638171816]

Archive:  rusvectores.zip
  inflating: rusvectores/meta.json   
  inflating: rusvectores/model.bin   
  inflating: rusvectores/model.txt   
  inflating: rusvectores/README      


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def text_to_vector(texts, model, vector_size):
    vectors = []
    for text in texts:
        words = text.split()
        word_vectors = [model[word] for word in words if word in model]

        if word_vectors:
            vectors.append(np.mean(word_vectors, axis=0))
        else:
            vectors.append(np.zeros(vector_size))

    return np.array(vectors)

In [17]:
  # Векторизация w2v
  train_w2v = text_to_vector(train_texts, w2v_model.wv, 100)
  val_w2v = text_to_vector(val_texts, w2v_model.wv, 100)

  # Векторизация с navec
  train_navec = text_to_vector(train_texts, navec_model, 300)
  val_navec = text_to_vector(val_texts, navec_model, 300)

  # Векторизация rusvectores
  train_rusvectores = text_to_vector(train_texts, rusvectores_model, 300)
  val_rusvectores = text_to_vector(val_texts, rusvectores_model, 300)

  # Обучение на w2v
  lr_w2v = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
  lr_w2v.fit(train_w2v, train_labels)
  pred_w2v = lr_w2v.predict(val_w2v)

  # Обучение на navec
  lr_navec = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
  lr_navec.fit(train_navec, train_labels)
  pred_navec = lr_navec.predict(val_navec)

  # Обучение на rusvectores
  lr_rusvectores = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
  lr_rusvectores.fit(train_rusvectores, train_labels)
  pred_rusvectores = lr_rusvectores.predict(val_rusvectores)

In [18]:
# Оценка
results = {
    "w2v": {
        "accuracy": accuracy_score(val_labels, pred_w2v),
        "f1": f1_score(val_labels, pred_w2v, average="weighted"),
    },
    "navec": {
        "accuracy": accuracy_score(val_labels, pred_navec),
        "f1": f1_score(val_labels, pred_navec, average="weighted"),
    },
    "rusvectores": {
        "accuracy": accuracy_score(val_labels, pred_rusvectores),
        "f1": f1_score(val_labels, pred_rusvectores, average="weighted"),
    },
}

results

{'w2v': {'accuracy': 0.7471, 'f1': 0.7343630107250386},
 'navec': {'accuracy': 0.7438, 'f1': 0.732203442415376},
 'rusvectores': {'accuracy': 0.2166, 'f1': 0.07712569455860596}}

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

best_model = navec_model  #Navec дал лучшие результаты
vector_size = 300

tfidf = TfidfVectorizer()
tfidf.fit(train_texts)

idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def text_to_tfidf_vector(texts, model, vector_size, idf_dict):
    vectors = []
    for text in texts:
        words = text.split()
        word_vectors = []
        word_weights = []

        for word in words:
            if word in model and word in idf_dict:
                word_vectors.append(model[word] * idf_dict[word])
                word_weights.append(idf_dict[word])

        if word_vectors:
            vectors.append(np.average(word_vectors, axis=0, weights=word_weights))
        else:
            vectors.append(np.zeros(vector_size))

    return np.array(vectors)

In [20]:
train_tfidf = text_to_tfidf_vector(train_texts, best_model, vector_size, idf_dict)
val_tfidf = text_to_tfidf_vector(val_texts, best_model, vector_size, idf_dict)

lr_tfidf = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr_tfidf.fit(train_tfidf, train_labels)
pred_tfidf = lr_tfidf.predict(val_tfidf)

results_tfidf = {
    "accuracy": accuracy_score(val_labels, pred_tfidf),
    "f1": f1_score(val_labels, pred_tfidf, average="weighted"),
}

results_tfidf

{'accuracy': 0.7322, 'f1': 0.7243934606017938}

In [21]:
test_tfidf = text_to_tfidf_vector(test_texts, best_model, vector_size, idf_dict)

pred_test = lr_tfidf.predict(test_tfidf)

final_results = {
    "accuracy": accuracy_score(test_labels, pred_test),
    "f1": f1_score(test_labels, pred_test, average="weighted"),
}

final_results

{'accuracy': 0.7325, 'f1': 0.7237976991106633}